# Module 10: Monitoring & Drift Detection

This notebook demonstrates:
- data drift monitoring (PSI/KS)
- performance monitoring (rolling WAPE/MAE)
- retraining trigger decision

It uses:
- latest forecast file in `outputs/forecasts/`
- actuals from `data/raw/sample_sales.csv`

Outputs are written to `outputs/reports/`.


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

import sys
sys.path.append('..')

from src.data.loaders import load_sales_data
from src.monitoring.drift import drift_report
from src.monitoring.performance import rolling_error_report

print('Imports OK')


Imports OK


In [ ]:
# Load latest forecasts
forecasts_dir = Path('../outputs/forecasts')
forecast_files = sorted(forecasts_dir.glob('*_sku_forecasts.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not forecast_files:
    raise FileNotFoundError('No forecast files found in outputs/forecasts')

fc_path = forecast_files[0]
fc = pd.read_csv(fc_path, parse_dates=['date'])
print('Forecast file:', fc_path.name, fc.shape)
fc.head()


In [ ]:
# Load actuals
raw_path = Path('../data/raw/sample_sales.csv')
df = load_sales_data(raw_path)
print('Actuals:', df.shape)

# Join on sku/date
actuals = df[['sku_id','date','units_sold']].rename(columns={'units_sold':'y_true'})
scored = fc.merge(actuals, on=['sku_id','date'], how='left')

print('Scored:', scored.shape)
print('Missing actuals rows:', scored['y_true'].isna().sum())
scored.dropna(subset=['y_true']).head()


In [ ]:
# Performance monitoring (rolling)
perf = rolling_error_report(
    scored.dropna(subset=['y_true']),
    date_col='date',
    y_true_col='y_true',
    y_pred_col='y_pred',
    window_days=28,
)
perf.tail()


In [ ]:
# Drift monitoring (compare last 90 days vs prior 90 days)
df['date'] = pd.to_datetime(df['date'])
as_of = df['date'].max()
cur_start = as_of - pd.Timedelta(days=90)
ref_start = cur_start - pd.Timedelta(days=90)

ref = df[(df['date'] > ref_start) & (df['date'] <= cur_start)]
cur = df[(df['date'] > cur_start) & (df['date'] <= as_of)]

numeric_cols = [c for c in ['price','stock_available','units_sold'] if c in df.columns]
drift = drift_report(ref, cur, numeric_cols=numeric_cols, buckets=10)
drift


In [ ]:
# Retraining trigger decision (simple thresholds)
psi_threshold = 0.25
wape_threshold = 0.45

max_psi = float(drift['psi'].max()) if len(drift) else float('nan')
latest_wape = float(perf['rolling_wape'].dropna().iloc[-1]) if len(perf) else float('nan')

trigger = (not np.isnan(max_psi) and max_psi >= psi_threshold) or (not np.isnan(latest_wape) and latest_wape >= wape_threshold)

report = {
    'as_of': str(as_of),
    'forecast_file': fc_path.name,
    'drift': {'numeric_cols': numeric_cols, 'max_psi': max_psi, 'psi_threshold': psi_threshold},
    'performance': {'latest_rolling_wape': latest_wape, 'wape_threshold': wape_threshold},
    'decision': {'trigger_retrain': bool(trigger)}
}

report


In [ ]:
# Save artifacts
out_dir = Path('../outputs/reports')
out_dir.mkdir(parents=True, exist_ok=True)

run_id = pd.Timestamp.utcnow().strftime('%Y%m%dT%H%M%SZ')

(out_dir / f'{run_id}_module10_drift.csv').write_text(drift.to_csv(index=False), encoding='utf-8')
(out_dir / f'{run_id}_module10_performance.csv').write_text(perf.to_csv(index=False), encoding='utf-8')
(out_dir / f'{run_id}_module10_notebook_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

print('Saved module10 artifacts with run_id:', run_id)
